# Pass@K Evaluation on BooIQ
It can evaluate the two paired model (SFT and RL) at one time.
## Code structure
1. Install the neccessary library
2. Global Configuration
3. Path Direction
4. Build prompt
5. extract function
6. pass@k function
7. load model
8. generate answers
9. Evluation pass@k function
10. Evaluation and store the results
11. Print out results

# BooIQ Dataset
A non-reasoning dataset. It is a question answering dataset for yes/no questions. Each example is a triplet of (question, passage, answer), with the title of the page as optional additional context. The text-pair classification setup is similar to existing natural language inference tasks.

## Example
Question: do iran and afghanistan speak the same language\
Answer: true\
Passage: Persian (/ˈpɜːrʒən, -ʃən/), also known by its endonym Farsi (فارسی fārsi (fɒːɾˈsiː) ( listen)), is one of the Western Iranian languages within the Indo-Iranian branch of the Indo-European language family. It is primarily spoken in Iran, Afghanistan (officially known as Dari since 1958), and Tajikistan (officially known as Tajiki since the Soviet era), and some other regions which historically were Persianate societies and considered part of Greater Iran. It is written in the Persian alphabet, a modified variant of the Arabic script, which itself evolved from the Aramaic alphabet.

In [ ]:
# Install libraries
!pip install -q transformers datasets accelerate bitsandbytes tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.4 MB/s eta 0:00:00


In [ ]:
# 1. Import libraries
import re, json, math, torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 2. Global configuration
DATASET_NAME = "google/boolq"
SPLIT = "validation"

MAX_EXAMPLES = 500   # number of data for testing
TEMPERATURE = 0.6
TOP_P = 0.95
MAX_NEW_TOKENS = 16  # only

# 3. Path of output json file
OUT_PATH = "boolq_deepseek_math_results.json"
# The models we will use for BooIQ
MODELS = [{"name": "deepseek-ai/deepseek-math-7b-instruct",
        "n_samples": 8,
        "k_list": [1, 8],
    },{"name": "deepseek-ai/deepseek-math-7b-rl",
        "n_samples": 1,
        "k_list": [1],}
]

# 4. Build prompt (strict instruction)
def build_prompt(ex):
    passage = ex["passage"]
    question = ex["question"]

    return f"""Read the passage and answer the question with only Yes or No.
            Passage:
            {passage}
            Question:
            {question}

            Answer:"""

# 5. Extract the answer
def normalize_pred(text):
    text = text.strip().lower()

    # only inspect beginning to avoid later contradictions
    head = text[:80]

    if re.search(r"\byes\b", head):
        return True
    if re.search(r"\bno\b", head):
        return False
    if head.startswith("true"):
        return True
    if head.startswith("false"):
        return False

    return None

# 6. Unbiased pass@k function
def pass_at_k(n, c, k):
    if c == 0:
        return 0.0
    if n - c < k:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)


# 7. function of loading model with 4-bit quantitizing for T4 GPU
def load_model(model_name):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model.eval()
    return tokenizer, model

# 8. Function of Generate answers by batch
@torch.no_grad()
def generate_batch(prompts, tokenizer, model, n_samples):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        do_sample=(n_samples > 1),
        temperature=TEMPERATURE if n_samples > 1 else None,
        top_p=TOP_P if n_samples > 1 else None,
        num_return_sequences=n_samples,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_len = inputs["input_ids"].shape[1]
    gen_texts = tokenizer.batch_decode(
        outputs[:, input_len:],
        skip_special_tokens=True,
    )

    grouped = []
    for i in range(len(prompts)):
        grouped.append(gen_texts[i*n_samples:(i+1)*n_samples])

    return grouped

# 9. Pass@k Evaluation Pipeling
def evaluate_model(model_name, examples, n_samples, k_list, batch_size=4):
    # Initialize the model in a function
    tokenizer, model = load_model(model_name)

    rows = []
    total_correct_for_pass1 = 0

    for start in tqdm(range(0, len(examples), batch_size), desc=model_name):
        batch = examples[start:start+batch_size]
        prompts = [x["prompt"] for x in batch]
        # generate answers by batch
        grouped_outputs = generate_batch(prompts, tokenizer, model, n_samples)
        # decide if the answer is correct in a batch
        for ex, outputs in zip(batch, grouped_outputs):
            gold = ex["answer"]
            preds = []
            correct_flags = []

            for text in outputs:
                pred = normalize_pred(text)
                correct = (pred is not None and pred == gold)

                preds.append({
                    "text": text,
                    "pred": pred,
                    "correct": correct,
                })
                correct_flags.append(correct)

            c = sum(correct_flags)

            row = {
                "id": ex["id"],
                "question": ex["question"],
                "gold": gold,
                "num_correct": c,
                "n_samples": n_samples,
                "samples": preds,
            }
            # Calculate pass@k for one questions
            for k in k_list:
                row[f"pass@{k}"] = pass_at_k(n_samples, c, k)

            rows.append(row)

    summary = {
        "model": model_name,
        "num_examples": len(rows),
        "n_samples": n_samples,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
    }

    for k in k_list:
        summary[f"pass@{k}"] = sum(r[f"pass@{k}"] for r in rows) / len(rows)
    # move the previous model and save space for the next model
    del model
    torch.cuda.empty_cache()

    return summary, rows

In [ ]:
# 10. Load BooIQ dataset
ds = load_dataset(DATASET_NAME, split=SPLIT)

if MAX_EXAMPLES is not None:
    ds = ds.select(range(min(MAX_EXAMPLES, len(ds))))

examples = []
# Formatting the dataset
for i, ex in enumerate(ds):
    examples.append({
        "id": i,
        "question": ex["question"],
        "passage": ex["passage"],
        "answer": ex["answer"],  # Bool: True / False
        "prompt": build_prompt(ex),
    })

print("Loaded examples:", len(examples))
print(examples[0]["prompt"])
print("Gold:", examples[0]["answer"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.69M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9427 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3270 [00:00<?, ? examples/s]

Loaded examples: 500
Read the passage and answer the question with only Yes or No.

Passage:
All biomass goes through at least some of these steps: it needs to be grown, collected, dried, fermented, distilled, and burned. All of these steps require resources and an infrastructure. The total amount of energy input into the process compared to the energy released by burning the resulting ethanol fuel is known as the energy balance (or ``energy returned on energy invested''). Figures compiled in a 2007 report by National Geographic Magazine point to modest results for corn ethanol produced in the US: one unit of fossil-fuel energy is required to create 1.3 energy units from the resulting ethanol. The energy balance for sugarcane ethanol produced in Brazil is more favorable, with one unit of fossil-fuel energy required to create 8 from the ethanol. Energy balance estimates are not easily produced, thus numerous such reports have been generated that are contradictory. For instance, a separa

In [ ]:
# 11. Evaluating pass@k for 2 models
all_results = {}

for cfg in MODELS:
  # evaluating the function (including loading model and calculating pass@k)
    summary, rows = evaluate_model(
        model_name=cfg["name"],
        examples=examples,
        n_samples=cfg["n_samples"],
        k_list=cfg["k_list"],
        batch_size=4,
    )
    # save the results
    all_results[cfg["name"]] = {
        "summary": summary,
        "rows": rows,
    }

    print("\n", cfg["name"])
    print(json.dumps(summary, indent=2))

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print("Saved to:", OUT_PATH)

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

deepseek-ai/deepseek-math-7b-rl: 100%|██████████| 125/125 [03:43<00:00,  1.78s/it]


 deepseek-ai/deepseek-math-7b-rl
{
  "model": "deepseek-ai/deepseek-math-7b-rl",
  "num_examples": 500,
  "n_samples": 1,
  "temperature": 0.6,
  "top_p": 0.95,
  "pass@1": 0.192
}
Saved to: boolq_deepseek_math_results.json


In [ ]:
# 12. Print out the results and calculate the reasoning index and RL improvement
sft = all_results["deepseek-ai/deepseek-math-7b-instruct"]["summary"]
rl = all_results["deepseek-ai/deepseek-math-7b-rl"]["summary"]

print("SFT pass@1:", sft["pass@1"])
print("SFT pass@8:", sft["pass@8"])
print("SFT reasoning index:", sft["pass@8"] - sft["pass@1"])
print("RL pass@1:", rl["pass@1"])
print("RL improvement:", rl["pass@1"] - sft["pass@1"])

SFT pass@1: 0.17325
SFT pass@8: 0.238
SFT diversity gap: 0.06475
RL pass@1: 0.192
RL improvement: 0.018750000000000017
